# SemiTNet — بازتولید، شبیه‌سازی و ارزیابی یکپارچه
این **تنها نوت‌بوک اصلی پروژه** است و کل مسیر را پوشش می‌دهد:
`Environment → Repository → Dataset → Paper-to-Code → Simulation → Results → Tables 1–3 → Figures 1–5 → Reproducibility`.

برای هر خروجی ارزیابی: **هدف، کد، توضیح فارسی، خروجی/مسیر artifact** ارائه می‌شود. مقادیر مقاله «مرجع مقاله» و مقادیر اندازه‌گیری‌شده «اجرای ارزیابی فعلی» هستند؛ هیچ baseline prediction ساخته نمی‌شود.

## 1) محیط و کنترل اجرا
فلگ‌ها: `RUN_SAFE` برای audit/export، `RUN_EVALUATION` برای اجرای ارزیابی، `RUN_HASH` برای SHA کامل و `RUN_FULL_26250` برای full training. پیش‌فرض همه خاموش است.

In [ ]:
# هدف: تعیین root و بررسی محیط
# ورودی: محیط Python و repository
# خروجی: نسخه‌ها، GPU، dependencies و Git HEAD
# رفتار مورد انتظار: read-only و بدون نصب خودکار
from pathlib import Path
import os,sys,json,subprocess,hashlib,importlib.util,platform,shlex
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from PIL import Image as PILImage
from IPython.display import display, Image
ROOT=next((p for p in [Path.cwd().resolve(),*Path.cwd().resolve().parents] if (p/'project.py').is_file() and (p/'reproduction/reference_contract.json').is_file()),None)
if ROOT is None: raise RuntimeError('repository root not found')
os.chdir(ROOT); RUN_SAFE=False; RUN_EVALUATION=False; RUN_HASH=False; RUN_FULL_26250=False
print('ROOT',ROOT); print('Python',sys.version.replace('\n',' ')); print('Platform',platform.platform())
for n in ['torch','numpy','pandas','matplotlib','PIL','yaml','detectron2','pycocotools']: print(n,importlib.util.find_spec(n) is not None)
try:
 import torch; print('PyTorch',torch.__version__,'CUDA',torch.cuda.is_available(),'GPU',torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')
except Exception as e: print(e)
print('HEAD',subprocess.run(['git','rev-parse','HEAD'],text=True,capture_output=True).stdout.strip())

## 2) ساختار مخزن
`scripts/` اجراها، `configs/` تنظیمات، `data/` ورودی، `outputs/` نتایج، `reproduction/` قرارداد علمی، `paper/reference/` مقادیر مرجع و `notebooks/` همین گزارش واحد است.

In [ ]:
# هدف: نمایش tree محدود repository
# ورودی: ROOT
# خروجی: مسیرهای مهم تا عمق دو
# رفتار مورد انتظار: dataset بزرگ truncate شود
for n in ['scripts','configs','data','outputs','reproduction','paper','notebooks']:
 b=ROOT/n; print('\n#',n,b.exists())
 if b.exists():
  for p in [x for x in sorted(b.rglob('*')) if len(x.relative_to(b).parts)<=2][:70]: print('  '*(len(p.relative_to(b).parts)-1)+('DIR ' if p.is_dir() else 'FILE ')+p.name)

## 3) مستندسازی Dataset
مسیرهای authoritative: `data/train/`، `data/test/` و `data/TED3-unlabeled-data-15k-pseudo-mask/`. pseudo-mask بدون provenance برابر human ground truth نیست. SHA overlap فقط با `RUN_HASH=True` اجرا می‌شود.

In [ ]:
# هدف: inventory dataset و leakage audit اختیاری
# ورودی: train/test/unlabeled
# خروجی: count/size/extensions/sample/manifest/overlap
# رفتار مورد انتظار: read-only؛ hash کامل opt-in
from collections import Counter
D={'train':ROOT/'data/train','test':ROOT/'data/test','unlabeled':ROOT/'data/TED3-unlabeled-data-15k-pseudo-mask'}; EXT={'.jpg','.jpeg','.png','.bmp','.tif','.tiff'}
rows=[]
for r,p in D.items():
 f=[x for x in p.rglob('*') if x.is_file()] if p.is_dir() else []
 rows.append([r,str(p.relative_to(ROOT)),p.is_dir(),len(f),round(sum(x.stat().st_size for x in f)/1024**3,3),dict(Counter(x.suffix.lower() for x in f).most_common(5))])
display(pd.DataFrame(rows,columns=['role','path','exists','files','GiB','extensions']))
for r,p in D.items():
 im=next((x for x in sorted(p.rglob('*')) if x.is_file() and x.suffix.lower() in EXT),None) if p.is_dir() else None
 if im: plt.figure(figsize=(8,3)); plt.imshow(PILImage.open(im),cmap='gray'); plt.title(str(im.relative_to(ROOT))); plt.axis('off'); plt.show()
if RUN_HASH:
 def H(x):
  h=hashlib.sha256(); f=x.open('rb')
  for z in iter(lambda:f.read(8*1024*1024),b''): h.update(z)
  f.close(); return h.hexdigest()
 hs={r:{H(x) for x in p.rglob('*') if x.is_file() and x.suffix.lower() in EXT} for r,p in D.items()}
 for a,b in [('train','test'),('train','unlabeled'),('test','unlabeled')]: print(a,b,len(hs[a]&hs[b]))

## 4) نگاشت مقاله به کد
هدف publication: 32 class، 1024 RGB، 100 query، 9 decoder layer، AdamW، LR=1e-4، steps 24000/25000 و 26,250 iteration. evaluator مقاله در `compute_article_metrics.py` و full teacher/student در `run_full.py` است.

In [ ]:
# هدف: نمایش config و paper-to-code mapping
# ورودی: reference_contract.json و configs/paper.yaml
# خروجی: جدول اجزای روش و محل پیاده‌سازی
# رفتار مورد انتظار: مسیرهای اجرایی مختلف با هم خلط نشوند
contract=json.loads((ROOT/'reproduction/reference_contract.json').read_text())
try:
 import yaml; cfg=yaml.safe_load((ROOT/'configs/paper.yaml').read_text())
except Exception: cfg={}
display(pd.DataFrame([{'parameter':k,'value':v} for k,v in cfg.items()]))
M=[['Encoder/Backbone','QuickSemiTransformer + full vendor','run_quick_real_experiment.py / vendor'],['Queries/Decoder','100 queries / 9 layers target','configs/paper.yaml + upstream'],['Loss','CE+Dice / upstream losses','supervised_loss + vendor'],['Teacher/Student','pseudo-label + student + EMA','quick script / run_full.py'],['Checkpoint eval','eval-only + COCO metrics','run_official_inference.py'],['Metrics','COCOeval@.5 + Dice/P/R/F1','compute_article_metrics.py'],['Figures/Tables','paper-style exporter','export_paper_style_figures.py']]
display(pd.DataFrame(M,columns=['Component','Implementation','Location']))

## 5) Workflow اجرا
`Data → Teacher → Loss → Pseudo Labels → Student → Ls+Lu → EMA → Evaluation → Tables/Figures`.
دستورات: `project.py ted3-preflight`، `project.py audit`، `run_quick_real_experiment.py`، `export_paper_style_figures.py`، `run_official_inference.py` و برای full training: `run_full.py`.

In [ ]:
# هدف: runner کنترل‌شده مراحل simulation/evaluation
# ورودی: RUN_* flags
# خروجی: command status و artifact matrix
# رفتار مورد انتظار: full training فقط opt-in
CMDS={'preflight':[sys.executable,'project.py','ted3-preflight'],'audit':[sys.executable,'project.py','audit'],'evaluation':[sys.executable,'scripts/run_quick_real_experiment.py'],'figures':[sys.executable,'scripts/export_paper_style_figures.py'],'full':[sys.executable,'scripts/run_full.py','--dataset','data/processed/ted3']}
def run(n):
 p=subprocess.run(CMDS[n],text=True,capture_output=True); print(' '.join(map(shlex.quote,CMDS[n])),p.returncode); print(p.stdout[-1800:]); return p.returncode
if RUN_SAFE: run('preflight'); run('audit'); run('figures')
if RUN_EVALUATION: run('evaluation'); run('figures')
if RUN_FULL_26250: run('full')
display(pd.DataFrame([['Preflight','project.py ted3-preflight','outputs/ted3_reproduction'],['Audit','project.py audit','audit/data processed'],['Checkpoint','run_official_inference.py','outputs/inference'],['Training','run_full.py','outputs/full'],['Metrics','compute_article_metrics.py','metrics/tables'],['Exports','export_paper_style_figures.py','outputs/paper_style']],columns=['Phase','Entrypoint','Artifact']))

## 6) بارگذاری نتایج موجود
فقط JSON واقعی خوانده می‌شود؛ missing result ساخته نمی‌شود. manifestها برای provenance فهرست می‌شوند.

In [ ]:
# هدف: catalog متریک‌ها و manifestها
# ورودی: outputs موجود
# خروجی: metrics_catalog و manifest_catalog
# رفتار مورد انتظار: فقط JSON واقعی
MET=['iou','dice','precision','recall','f1']; rec=[]; mans=[]
for b in [ROOT/'outputs/ted3_reproduction',ROOT/'outputs/paper_reproduction',ROOT/'outputs/inference',ROOT/'outputs/final']:
 if b.exists():
  for p in b.rglob('*.json'):
   try:o=json.loads(p.read_text())
   except:continue
   if isinstance(o,dict) and all(isinstance(o.get(k),(int,float)) for k in MET): rec.append({'source':str(p.relative_to(ROOT)),**{k:float(o[k]) for k in MET}})
   if 'manifest' in p.name.lower(): mans.append([str(p.relative_to(ROOT)),list(o)[:10] if isinstance(o,dict) else []])
display(pd.DataFrame(rec)); display(pd.DataFrame(mans,columns=['manifest','keys']))

# 7) بخش ارزیابی مقاله — Tables 1–3 و Figures 1–5
هر زیر‌بخش شامل توضیح فارسی، کد و artifact است. `PAPER` منبع reference و `FINAL` منبع خروجی اندازه‌گیری‌شده است.

In [ ]:
# هدف: آماده‌سازی منابع مشترک بخش ارزیابی
# ورودی: paper/reference و outputs/final
# خروجی: DataFrameها و metrics مورد استفاده در تمام خروجی‌ها
# رفتار مورد انتظار: مرجع مقاله و measured result جدا بمانند
PAPER=ROOT/'paper/reference'; FINAL=ROOT/'outputs/final'; PAPER_STYLE=ROOT/'outputs/paper_style'
table1=pd.read_csv(PAPER/'table1_dataset.csv'); table2=pd.read_csv(PAPER/'table2_overall.csv'); table3=pd.read_csv(PAPER/'table3_groups.csv')
metrics=json.loads((FINAL/'metrics.json').read_text()); history=pd.read_csv(FINAL/'history.csv')
print({k:round(metrics[k],4) for k in MET})

## 7.1 Table 1 — توزیع داده
**توضیح:** جدول توصیفی dataset از `table1_dataset.csv` خوانده می‌شود و متریک مدل نیست.

In [ ]:
# هدف: نمایش Table 1
# ورودی: table1_dataset.csv
# خروجی: توزیع dataset
# رفتار مورد انتظار: مقادیر مرجع بدون تغییر
display(table1)

![Table 1](../outputs/paper_style/tables/table_01_tsi15k_distribution.png)

## 7.2 Table 2 — عملکرد کلی
**توضیح:** baselineها فقط از مرجع مقاله؛ اجرای فعلی جداگانه با IoU/Dice/Precision/Recall/F1 نمایش داده می‌شود.

In [ ]:
# هدف: نمایش Table 2 و measured metrics جداگانه
# ورودی: table2_overall.csv و metrics.json
# خروجی: reference table + current evaluation row
# رفتار مورد انتظار: عدم نسبت دادن measured numbers به baselineها
display(table2.rename(columns={'model':'Model','iou':'IoU','dice':'Dice','precision':'Precision','recall':'Recall','f1':'F1','params_m':'Params (M)'}))
display(pd.DataFrame([{'Run':'اجرای ارزیابی فعلی',**{k.upper():metrics[k] for k in MET}}]))

![Table 2](../outputs/paper_style/tables/table_02_overall_performance.png)

## 7.3 Table 3 — زیرگروه‌ها
**توضیح:** fully dentate و partially edentulous از `table3_groups.csv`. اگر measured subgroup artifact نباشد مقدار جدید ساخته نمی‌شود.

In [ ]:
# هدف: نمایش Table 3
# ورودی: table3_groups.csv
# خروجی: group-level reference metrics
# رفتار مورد انتظار: فقط داده موجود
display(table3.rename(columns={'group':'Group','model':'Model','iou':'IoU','dice':'Dice','precision':'Precision','recall':'Recall','f1':'F1'}))

![Table 3](../outputs/paper_style/tables/table_03_group_performance.png)

## 7.4 Figure 1 — Architecture
**توضیح:** `Panoramic → Encoder → Feature Pyramid → Query Initialization → Mask Decoder → Predictions`؛ تولید توسط `figure1()`.

In [ ]:
# هدف: نمایش Figure 1
# ورودی: architecture PNG
# خروجی: دیاگرام معماری
# رفتار مورد انتظار: نمایش artifact موجود
p=PAPER_STYLE/'figures/figure_01_semitnet_architecture.png'; display(Image(filename=str(p))) if p.exists() else print('Missing',p)

![Figure 1](../outputs/paper_style/figures/figure_01_semitnet_architecture.png)

## 7.5 Figure 2 — Teacher/Student
**توضیح:** pseudo-label از Teacher، آموزش Student با `Ls+Lu` و به‌روزرسانی EMA؛ تولید توسط `figure2()`.

In [ ]:
# هدف: نمایش Figure 2
# ورودی: workflow PNG
# خروجی: Teacher/Student/Pseudo-label/EMA diagram
# رفتار مورد انتظار: نمایش artifact موجود
p=PAPER_STYLE/'figures/figure_02_teacher_student_workflow.png'; display(Image(filename=str(p))) if p.exists() else print('Missing',p)

![Figure 2](../outputs/paper_style/figures/figure_02_teacher_student_workflow.png)

## 7.6 Figure 3 — Loss و Precision
**توضیح:** نمودار دو محوره مستقیماً از `history.csv`؛ هیچ نقطه مصنوعی اضافه نمی‌شود.

In [ ]:
# هدف: بازتولید Figure 3
# ورودی: history.csv
# خروجی: dual-axis Loss/Precision plot
# رفتار مورد انتظار: تمام نقاط از CSV واقعی
x=np.arange(1,len(history)+1); fig,ax1=plt.subplots(figsize=(11,6)); ax2=ax1.twinx()
ax1.plot(x,history['loss'],marker='o',label='Training Loss'); ax2.plot(x,history['precision'],marker='o',label='Precision (%)')
ax1.set_xlabel('Training Step'); ax1.set_ylabel('Training Loss'); ax2.set_ylabel('Precision (%)'); ax1.grid(alpha=.25); ax1.set_title('Loss and Precision vs. Training Step'); plt.show()

![Figure 3](../outputs/paper_style/figures/figure_03_training_loss_precision.png)

## 7.7 Figure 4 — Radar متریک‌ها
**توضیح:** پنج محور `IoU, Dice, Precision, Recall, F1`. شکل مرجع paper-style و رادار اجرای فعلی از `metrics.json` جدا هستند.

In [ ]:
# هدف: ساخت radar اجرای ارزیابی فعلی
# ورودی: metrics.json
# خروجی: radar پنج‌محوره
# رفتار مورد انتظار: فقط متریک‌های اندازه‌گیری‌شده
labels=['IoU','Dice','Precision','Recall','F1']; vals=[metrics[k.lower()] for k in labels]; a=np.linspace(0,2*np.pi,5,endpoint=False).tolist(); a+=a[:1]; v=vals+vals[:1]
fig,ax=plt.subplots(figsize=(7,6),subplot_kw={'projection':'polar'}); ax.set_theta_offset(np.pi/2); ax.set_theta_direction(-1); ax.set_thetagrids(np.degrees(a[:-1]),labels); ax.set_ylim(0,100); ax.plot(a,v); ax.fill(a,v,alpha=.15); ax.set_title('اجرای ارزیابی فعلی'); plt.show()

![Figure 4](../outputs/paper_style/figures/figure_04_performance_radar.png)

## 7.8 Figure 5 — خروجی کیفی واقعی
**توضیح:** فقط artifact واقعی نمایش داده می‌شود. برای Mask R-CNN، MPFormer، Mask2Former، MaskDINO و GEM prediction ساخته/حدس زده نمی‌شود.

In [ ]:
# هدف: نمایش Figure 5 واقعی
# ورودی: measured qualitative artifact یا predictions.png
# خروجی: تصویر کیفی موجود
# رفتار مورد انتظار: عدم ساخت placeholder به‌عنوان نتیجه
q=PAPER_STYLE/'figures/figure_05b_measured_qualitative_only.png'; s=FINAL/'figures/predictions.png'
if q.exists(): display(Image(filename=str(q)))
elif s.exists(): display(Image(filename=str(s)))
else: print('No measured qualitative artifact found')

## 8) مقایسه Paper و Measured
جدول زیر published metrics را کنار artifact اندازه‌گیری‌شده و source آن قرار می‌دهد.

In [ ]:
# هدف: Published/Measured/Gap
# ورودی: reference_contract و rec
# خروجی: comparison table
# رفتار مورد انتظار: missing measured خالی
pub=contract['publication']['published_metrics_percent']; prio=['semi_supervised','supervised','checkpoint_eval','outputs/inference','outputs/final']; sel=next((r for t in prio for r in rec if t in r['source']),None)
display(pd.DataFrame([[k.upper(),pub[k],None if not sel else sel[k],None if not sel else sel[k]-pub[k],None if not sel else sel['source']] for k in MET],columns=['Metric','Published SemiTNet','Measured Result','Gap','Source']))

## 9) Reproducibility
دستورات اصلی:
`python project.py ted3-preflight` · `python project.py audit` · `python scripts/run_quick_real_experiment.py` · `python scripts/export_paper_style_figures.py` · `python scripts/validate_final_outputs.py`.
برای full architecture از `run_full.py` و برای checkpoint سازگار از `run_official_inference.py` استفاده می‌شود.

In [ ]:
# هدف: نمایش audit و checklist
# ورودی: reproducibility_audit.json، manifests و artifacts
# خروجی: gate status و completeness checklist
# رفتار مورد انتظار: PASS/BLOCKED صریح نمایش داده شوند
a=ROOT/'outputs/reproducibility_audit.json'
if a.exists():
 o=json.loads(a.read_text()); display(pd.DataFrame([[k,v.get('status'),v.get('reason',v.get('allowed_claim',''))] for k,v in o.items() if isinstance(v,dict) and 'status' in v],columns=['Gate','Status','Reason']))
m=FINAL/'run_manifest.json'
if m.exists(): print('Seed',json.loads(m.read_text()).get('seed','not recorded'))
display(pd.DataFrame([['Dataset paths',all(p.exists() for p in D.values())],['Paper config',(ROOT/'configs/paper.yaml').exists()],['Reference contract',(ROOT/'reproduction/reference_contract.json').exists()],['Metrics',(FINAL/'metrics.json').exists()],['History',(FINAL/'history.csv').exists()],['Tables 1–3',all((PAPER/x).exists() for x in ['table1_dataset.csv','table2_overall.csv','table3_groups.csv'])],['Paper-style outputs',PAPER_STYLE.exists()]],columns=['Check','Status']))

## 10) جمع‌بندی نهایی
این یک نوت‌بوک واحد، محیط، dataset، paper-to-code، workflow، نتایج، **Tables 1–3، Figures 1–5، توضیح فارسی کد هر خروجی، مقایسه مرجع/اندازه‌گیری‌شده و reproducibility** را پوشش می‌دهد. provenance هر عدد/تصویر حفظ می‌شود و برای مدلی که اجرا نشده prediction ساخته نمی‌شود.